# Experiment 39 — Local-Contrastive SparseWalker from scratch

Same corrected plain SparseWalker v1.1 recurrence that beat SASRec on Amazon, but **no warm start, no optimizer, no backward**. This experiment removes pheromones entirely and tests whether the representation itself can self-organize with local contrastive/competitive rules.

Reference Beauty numbers: PheromoneWalker v1 val NDCG@10 ≈ 0.00235; SASRec test 0.03120; gradient SparseWalker test 0.04488.


In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=False)
import os, sys, shutil, subprocess, runpy, torch
from pathlib import Path
REPO='/content/Sparsewalker'; BRANCH='agent/local-contrastive-walker-v1'
if os.path.exists(REPO): shutil.rmtree(REPO)
subprocess.run(['git','clone','-q','-b',BRANCH,'https://github.com/hanialshater/Sparsewalker-.git',REPO],check=True)
for p in [f'{REPO}/src',f'{REPO}/experiments']: sys.path.insert(0,p) if p not in sys.path else None
import sparsewalker
assert torch.cuda.is_available(), 'GPU runtime required'
print('GPU',torch.cuda.get_device_name(0),'torch',torch.__version__,'bf16',torch.cuda.is_bf16_supported())
print('BRANCH',BRANCH,'PACKAGE',sparsewalker.__file__)


In [ ]:
SCRIPT=f'{REPO}/experiments/run_amazon_local_contrastive_walker.py'
text=Path(SCRIPT).read_text()
assert '.backward(' not in text
assert 'torch.optim' not in text
print('Backward/optimizer gates passed.')


In [ ]:
sys.argv=[SCRIPT,'--dataset','beauty','--epochs','20','--batch-size','512','--eval-every','1']
runpy.run_path(SCRIPT,run_name='__main__')


In [ ]:
import json, pandas as pd
root=Path('/content/drive/MyDrive/sparsewalker_local_contrastive/beauty/seed42')
hist=pd.DataFrame(json.loads((root/'history.json').read_text()))
display(hist[['epoch','mean_contrastive_margin','mean_positive_prob','mean_negative_prob','mean_router_confidence','mean_value_update','mean_context_error','val_NDCG@10','val_HR@10','positions_per_s']])
best=hist.loc[hist['val_NDCG@10'].idxmax()]
print('BEST',best.to_dict())
print('PHEROMONE_V1_VAL_NDCG',0.0023507147682577624)


## Read this result

The most informative comparison is **mean_contrastive_margin vs validation NDCG@10**. If margin rises while NDCG does not, the local contrastive representation is learning the wrong geometry. If both rise and this passes ~0.00235, it beats the best simple backward-free pheromone result without any engineered item→concept mapping.
